# Transcript Summarization with GPT

This notebook uses the OpenAI API to summarize YouTube video transcriptions related to anabolic steroid use.

For each video not yet processed, the model extracts structured information (topic, mentioned substances, side effects, etc.) and saves the result as JSON.

## 1. Imports and Configuration

Imports and definition of input (transcriptions), output (JSON summaries) and log file paths.

In [ ]:
import pandas as pd
import os
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal
import json
from tqdm import tqdm

PROJECT_ROOT = os.path.abspath(os.getcwd())

# Input and output directories (kept generic)
INPUT_TRANSCRIPTIONS_DIR = os.path.join(PROJECT_ROOT, 'transcriptions')
OUTPUT_SUMMARIES_DIR = os.path.join(PROJECT_ROOT, 'resumos_json')
SUMMARIES_LOG_DIR = os.path.join(PROJECT_ROOT, 'resumos')
os.makedirs(INPUT_TRANSCRIPTIONS_DIR, exist_ok=True)
os.makedirs(OUTPUT_SUMMARIES_DIR, exist_ok=True)
os.makedirs(SUMMARIES_LOG_DIR, exist_ok=True)

# Log file for processed ids (kept original name and added English alias)
arquivo_log = os.path.join(SUMMARIES_LOG_DIR, 'log_resumos.txt')
LOG_FILE = arquivo_log

## 2. Video IDs to Process

Load the predictions CSV and extract the list of unique `video_id`. Only videos present in this list will be summarized.

In [ ]:
MODEL_PREDICTIONS = os.path.join(PROJECT_ROOT, 'modelo', 'prediction', 'final_comments_match_cleaned_pred.csv')
df = pd.read_csv(MODEL_PREDICTIONS)
list_ids = (df['video_id'].unique()).tolist()

In [ ]:
len(list_ids)

## 3. OpenAI Client

Reads the API key from a local file and initializes the OpenAI client.

In [ ]:
# Load API key from summaries folder
api_key_file = os.path.join(SUMMARIES_LOG_DIR, 'api_key.txt')
with open(api_key_file, 'r') as f:
    api_key = f.read().split()[0]

client_gpt = OpenAI(api_key=api_key)

## 4. Esquema de Saída Estruturada (Pydantic)

Define o modelo `Resumo` com os campos que o GPT deve preencher para cada vídeo: tema, motivo de uso, substâncias, efeitos, tipo de conteúdo, público-alvo, autoridade percebida e resumo textual.

In [ ]:
class Resumo(BaseModel):
    tema_principal: str
    motivo_de_uso: Literal[
        "Médico/Terapêutico", 
        "Estético/Performance", 
        "Educativo/Não Especificado"
    ]
    esteroides_anabolizantes_mencionados: list[str]
    sintomas_e_efeitos_mencionados: list[str]
    menciona_fontes_cientificas: bool
    tipo_de_conteudo: Literal[
        "Experiência Pessoal", 
        "Explicação Técnica", 
        "Entrevista", 
        "Notícia", 
        "Debate", 
        "Propaganda", 
        "Outro"
    ]
    publico_alvo: Literal[
        "Atletas/Marombeiros", 
        "Pacientes", 
        "Médicos", 
        "Público Geral", 
        "Desconhecido"
    ]
    autoridade_percebida: Literal[
        "Médico", 
        "Atleta", 
        "Influenciador", 
        "Jornalista", 
        "Desconhecido"
    ]
    resumo: str

## 5. Summarization Prompt

Prompt template sent to GPT. Instructs the model to extract the schema fields from the video transcription, including normalization rules for lists of substances and symptoms. (The prompt content below is preserved as originally authored.)

In [ ]:
prompt_template = '''
Você é um(a) assistente para SUMARIZAÇÃO CIENTÍFICA de vídeos do YouTube, preciso de saídas CONSISTENTES e AVALIÁVEIS.

Analise o TEXTO_FONTE e extraia as seguintes informações de forma objetiva:

- "tema_principal": O assunto principal do vídeo em 2-5 palavras.

- "motivo_de_uso": Qual é o principal CONTEXTO ou MOTIVO para o uso das substâncias discutido no vídeo?
  Classifique em APENAS UMA das seguintes categorias:
  - "Médico/Terapêutico" (Se o foco for reposição hormonal, TRT, indicação médica, tratamento de hipogonadismo, etc.)
  - "Estético/Performance" (Se o foco for ganho de massa, força, estética, "shape", "maromba", performance, competição, etc.)
  - "Educativo/Não Especificado" (Se for um vídeo apenas informativo, histórico, ou se o motivo não for claro.)
  
- "esteroides_anabolizantes_mencionados": Uma lista Python de nomes de medicamentos, anabolizantes, esteroides ou substâncias (ex: "durateston", "testosterona", "hemogenin"). Se nenhum for mencionado, retorne uma lista vazia [].

- "sintomas_e_efeitos_mencionados": Uma lista Python de sintomas ou efeitos colaterais relacionados ao uso de esteroides e anabolizantes (ex: "acne", "queda de cabelo", "insonia", "ansiedade", "ganho de força"). Se nenhum for mencionado, retorne uma lista vazia [].

- "menciona_fontes_cientificas": O texto cita fontes científicas (estudos, artigos, pesquisas, etc.)?
Responda APENAS com `True` ou `False`.

- "tipo_de_conteudo": Classifique o formato do vídeo.
Responda APENAS com uma das opções: "Experiência Pessoal", "Explicação Técnica", "Entrevista", "Notícia", "Debate", "Propaganda", "Outro".

- "publico_alvo": Infira o público-alvo principal. 
Responda APENAS com uma das opções: "Atletas/Marombeiros", "Pacientes", "Médicos", "Público Geral", "Desconhecido".

- "autoridade_percebida": Se o falante se identificar, qual sua autoridade?
Responda APENAS com uma das opções: "Médico", "Atleta", "Influenciador", "Jornalista", "Desconhecido".

- "resumo": Resuma o conteúdo abaixo em até 150 palavras, enfatizando o tema principal e as ideias centrais.
Use linguagem neutra e informativa, sem expressar opiniões pessoais.
O resumo será usado para análise semântica e agrupamento de temas.

IMPORTANTE: 
Antes de gerar as listas finais de "sintomas_e_efeitos_mencionados" e "esteroides_anabolizantes_mencionados", normalize cada item seguindo estas regras:

- Sempre usar minúsculas.
- Evitar variações sinônimas; escolha apenas uma forma padronizada para cada substância ou sintoma.
- Sempre devolver no singular.
- Para sintomas/efeitos: sempre colocar o substantivo primeiro e o adjetivo depois (ex.: "libido baixa", "força aumentada").
- Para esteroides/substâncias: padronize nomes comerciais e nomes genéricos, mas NÃO invente termos nem traduções.
- Se dois itens forem semanticamente iguais, retorne apenas um.
- Remova duplicatas mesmo que escritas de formas diferentes.

TEXTO_FONTE:
'''

## 6. Progress Control (Log)

Helper function that reads the log file to return IDs of already processed videos, avoiding reprocessing.

In [ ]:
def load_log():
    if not os.path.exists(arquivo_log):
        return []
    with open(arquivo_log, 'r') as log_out:
        ids = log_out.read().split()
    return ids

## 7. Main Summarization Loop

Iterates over transcription files. For each video not yet processed (according to the log), it sends the transcription to the GPT model, saves the structured JSON result and records the ID in the log.

In [ ]:
for arquivo in tqdm(os.listdir(INPUT_TRANSCRIPTIONS_DIR)):
    nome_arquivo, _ = os.path.splitext(arquivo)
    ids_processados = load_log()
    
    if nome_arquivo in list_ids and nome_arquivo not in ids_processados:
        with open(os.path.join(INPUT_TRANSCRIPTIONS_DIR, arquivo), 'r') as f:
            texto = f.read()
            
            response_gpt = client_gpt.responses.parse(
                model='gpt-5-nano',
                input=prompt_template + texto,
                text_format=Resumo
            )
            
            output_file = os.path.join(OUTPUT_SUMMARIES_DIR, f'{nome_arquivo}.json')
            
            with open(arquivo_log, 'a') as log:
                log.write(nome_arquivo + '\n')
                
            with open(output_file, 'w', encoding='utf-8') as f_out:
                json.dump(
                    response_gpt.output_parsed.model_dump(),
                    f_out,
                    ensure_ascii=False,
                    indent=4
                )

## 8. Check Progress

Shows how many videos have been processed so far.

In [ ]:
len(ids_processados)